# 04b — Cross-City Vocab Unification

Runs ONCE, after both Bogor's and Warsaw's `04_tvg_construction` have
already completed independently. Each city built its own `highway_vocab`
/ `building_type_vocab` from its own OSM extract, so the same integer
currently means a DIFFERENT road/building type in each city's saved
`.pt` graphs. Pooling them as-is would silently misalign the shared
embedding tables in `06`/`07`.

This notebook does NOT recompute isovist geometry or refetch OSM data.
It only: (1) unions both cities' cached vocabularies into one, (2)
remaps the vocab-index tensors already saved inside every `.pt` TVG
graph, (3) writes the remapped graphs to a SEPARATE output directory
for you to spot-check before swapping them in, and (4) updates each
city's cached vocab JSON + `buildings.parquet` so any future rerun
(e.g. adding a third city) starts from the unified vocab already.

Nothing here touches Bogor's or Warsaw's raw source data or the
already-checkpointed per-point logs — this only patches the categorical
indices inside the saved graph tensors.

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric geopandas shapely pandas

In [ ]:
# ── Point directly at each city's existing base_dir ────────────────
# (Deliberately not going through configs/paths.yaml here — that file
# currently holds only ONE active city at a time. Hardcode both real
# locations from your Drive, matching what 04 already used for each.)
from pathlib import Path

BOGOR_BASE_DIR  = Path("/content/drive/MyDrive/crash-dualgraph/data")            # unprefixed, per original layout
WARSAW_BASE_DIR = Path("/content/drive/MyDrive/crash-dualgraph/data/warsaw")

CITY_DIRS = {
    "bogor":  {"interim": BOGOR_BASE_DIR / "interim",  "processed": BOGOR_BASE_DIR / "processed"},
    "warsaw": {"interim": WARSAW_BASE_DIR / "interim", "processed": WARSAW_BASE_DIR / "processed"},
}

for city, d in CITY_DIRS.items():
    cache_dir = d["interim"] / "osm_cache"
    assert cache_dir.exists(), f"{city}: expected OSM cache at {cache_dir}"
    assert (d["processed"] / "tvg_graphs").exists(), f"{city}: expected TVG graphs at {d['processed'] / 'tvg_graphs'}"
print("Both cities' 04 outputs found.")

In [ ]:
import vocab_merge as vm

hw_vocabs, bt_vocabs = {}, {}
for city, d in CITY_DIRS.items():
    cache_dir = d["interim"] / "osm_cache"
    hw_vocabs[city] = vm.load_vocab(cache_dir / "highway_vocab.json")
    bt_vocabs[city] = vm.load_vocab(cache_dir / "building_type_vocab.json")
    print(f"[{city}] highway_vocab: {len(hw_vocabs[city])} types | building_type_vocab: {len(bt_vocabs[city])} types")

In [ ]:
# ── Build the unified vocab ───────────────────────────────────────
unified_hw = vm.build_unified_vocab(*hw_vocabs.values(), fallback="unclassified")
unified_bt = vm.build_unified_vocab(*bt_vocabs.values(), fallback="unknown")

print(f"Unified highway_vocab: {len(unified_hw)} types (was {[len(v) for v in hw_vocabs.values()]} per city)")
print(f"Unified building_type_vocab: {len(unified_bt)} types (was {[len(v) for v in bt_vocabs.values()]} per city)")
print()
print("NOTE: embed_dim=4 (highway, was sized for vocab=13) and embed_dim=8-capped")
print("(building, was sized for vocab=58) in model.yaml were set against BOGOR-ONLY")
print("vocab sizes. Re-check both against the unified counts above before 06/07.")

In [ ]:
# ── Remap each city's saved graphs into a SEPARATE dir ────────────
# Never overwrites tvg_graphs/ directly — spot-check the *_unified copy
# below, THEN run the swap cell at the very end.
remap_stats = {}
for city, d in CITY_DIRS.items():
    tvg_dir = d["processed"] / "tvg_graphs"
    out_dir = d["processed"] / "tvg_graphs_unified"
    n_ok, n_err = vm.remap_city_graphs(
        tvg_dir=tvg_dir, out_dir=out_dir,
        old_highway_vocab=hw_vocabs[city], new_highway_vocab=unified_hw,
        old_building_vocab=bt_vocabs[city], new_building_vocab=unified_bt,
    )
    remap_stats[city] = (n_ok, n_err)

assert all(err == 0 for _, err in remap_stats.values()), "Fix errors above before continuing."

In [ ]:
# ── Spot-check: pick a few graphs per city, confirm old category string
# still resolves to the SAME string under the new index ────────────
import random
import torch

inv_unified_hw = {i: c for c, i in unified_hw.items()}
inv_unified_bt = {i: c for c, i in unified_bt.items()}

for city, d in CITY_DIRS.items():
    old_dir = d["processed"] / "tvg_graphs"
    new_dir = d["processed"] / "tvg_graphs_unified"
    inv_old_hw = {i: c for c, i in hw_vocabs[city].items()}
    inv_old_bt = {i: c for c, i in bt_vocabs[city].items()}

    sample = random.sample(sorted(old_dir.glob('*.pt')), min(3, len(list(old_dir.glob('*.pt')))))
    print(f"\n=== {city} ===")
    for p in sample:
        old_data = torch.load(p, weights_only=False)
        new_data = torch.load(new_dir / p.name, weights_only=False)

        old_hw_str = inv_old_hw[int(old_data['incident'].highway_type_idx.item())]
        new_hw_str = inv_unified_hw[int(new_data['incident'].highway_type_idx.item())]
        match = "✅" if old_hw_str == new_hw_str else "❌ MISMATCH"
        print(f"  {p.stem}: incident highway  old='{old_hw_str}' -> new='{new_hw_str}'  {match}")

        if old_data['building'].type_idx.numel():
            bi = 0
            old_bt_str = inv_old_bt[int(old_data['building'].type_idx[bi].item())]
            new_bt_str = inv_unified_bt[int(new_data['building'].type_idx[bi].item())]
            match = "✅" if old_bt_str == new_bt_str else "❌ MISMATCH"
            print(f"  {p.stem}: building[0] type  old='{old_bt_str}' -> new='{new_bt_str}'  {match}")

## Only run below once every spot-check above prints ✅

Swaps the unified graphs into `tvg_graphs/`, keeps the pre-unification
originals safely at `tvg_graphs_pre_unification_backup/`, and updates
each city's cached vocab JSON + `buildings.parquet` so this never needs
re-running unless a third city is added later.

In [ ]:
for city, d in CITY_DIRS.items():
    tvg_dir = d["processed"] / "tvg_graphs"
    remapped_dir = d["processed"] / "tvg_graphs_unified"
    vm.swap_in_remapped_graphs(tvg_dir, remapped_dir)

    cache_dir = d["interim"] / "osm_cache"
    vm.save_vocab(unified_hw, cache_dir / "highway_vocab.json")
    vm.save_vocab(unified_bt, cache_dir / "building_type_vocab.json")
    vm.remap_buildings_parquet_type_idx(cache_dir / "buildings.parquet", bt_vocabs[city], unified_bt)
    print(f"[{city}] vocab cache + buildings.parquet updated to unified vocab.\n")

print("Done. Both cities' tvg_graphs/ now share one global vocab.")
print("Next: proceed to 05_dataset_assembly per city, then the new pooled/")
print("per-city comparison tracks in 07.")